# End-to-End EDA & Data Preprocessing — 30-Minute Jupyter Notes

**Goal:** Understand the complete workflow from problem definition to preprocessing, with simple explanations and practical Python examples.

### 30-minute roadmap
| Time | Topic |
|---|---|
| 0–2 min | Problem Definition |
| 2–4 min | Data Collection |
| 4–6 min | Understanding Dataset |
| 6–10 min | EDA |
| 10–13 min | Data Cleaning |
| 13–16 min | Outliers |
| 16–20 min | Feature Engineering |
| 20–22 min | Feature Selection |
| 22–24 min | Feature-Target Separation |
| 24–28 min | Preprocessing |
| 28–30 min | Data Leakage + Final Pipeline |

> **Important:** In a real ML project, the exact order can vary. In particular, train/test splitting should happen early enough to prevent information from the test set leaking into preprocessing decisions.

## 1. Problem Definition

Before touching the dataset, clearly define the business/ML problem.

Ask:
- What are we trying to predict or understand?
- What is the **target variable**?
- Is it classification, regression, clustering, or another task?
- What is the business objective?
- Which evaluation metric should be used?

### Example
**Business problem:** Predict whether a customer will default on a loan.

- Target: `default`
- Features: income, age, loan amount, credit score, etc.
- Problem type: Classification
- Possible metrics: Accuracy, Precision, Recall, F1-score, ROC-AUC

### Why it matters
A poorly defined problem can lead to the wrong target, wrong features, and wrong evaluation metric.

## 2. Data Collection

Data can come from:
- CSV/Excel files
- SQL databases
- APIs
- Cloud storage
- Web data
- Sensors/logs
- Public datasets

### Good practice
Document:
- Source of data
- Collection date
- Number of records
- Data dictionary
- Meaning of each feature
- Possible privacy or quality issues

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Example:
# df = pd.read_csv("data.csv")

# Demo dataset
df = pd.DataFrame({
    "Age": [22, 25, 30, 35, 40, 45, 50, 55, 60, 65],
    "Income": [25000, 30000, 35000, 42000, 50000, 58000, 65000, 72000, 85000, 95000],
    "LoanAmount": [5000, 7000, 9000, 12000, 15000, 18000, 22000, 25000, 30000, 35000],
    "CreditScore": [650, 670, 690, 710, 720, 730, 750, 760, 780, 800],
    "Default": [1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
})

df.head()

## 3. Understanding the Dataset

The first step after loading data is to understand its structure.

### Important functions

- `head()` → first rows
- `tail()` → last rows
- `shape` → rows and columns
- `columns` → column names
- `info()` → data types and non-null counts
- `describe()` → numerical summary
- `nunique()` → number of unique values
- `value_counts()` → frequency of categories

### Questions to answer
1. How many rows and columns are there?
2. Which columns are numerical?
3. Which columns are categorical?
4. Are there missing values?
5. Is the target balanced?
6. Are there suspicious values?

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nInfo:")
df.info()

print("\nDescriptive statistics:")
display(df.describe())

print("\nUnique values:")
display(df.nunique())

# 4. Exploratory Data Analysis (EDA)

EDA is the process of exploring data to discover:
- Patterns
- Relationships
- Distributions
- Missing values
- Outliers
- Class imbalance
- Unexpected values

EDA generally includes **univariate, bivariate, and multivariate analysis**.

## 4.1 Univariate Analysis

**One variable at a time.**

For numerical variables:
- Mean, median, standard deviation
- Histogram
- KDE/density plot
- Box plot

For categorical variables:
- Frequency
- Percentage
- Bar chart

### Example questions
- What is the distribution of income?
- What is the most common category?
- Is the variable skewed?

In [ ]:
# Numerical summary
display(df[["Age", "Income", "LoanAmount", "CreditScore"]].describe())

# Histogram
df["Income"].hist(bins=10)
plt.title("Income Distribution")
plt.xlabel("Income")
plt.ylabel("Frequency")
plt.show()

# Box plot
sns.boxplot(x=df["LoanAmount"])
plt.title("Loan Amount - Box Plot")
plt.show()

## 4.2 Bivariate Analysis

**Two variables at a time.**

Examples:
- Numerical vs numerical → scatter plot/correlation
- Categorical vs numerical → box plot/grouped statistics
- Categorical vs categorical → crosstab/bar chart

### Example
Does income have a relationship with loan amount?

In [ ]:
sns.scatterplot(data=df, x="Income", y="LoanAmount", hue="Default")
plt.title("Income vs Loan Amount")
plt.show()

print("Correlation:")
print(df[["Income", "LoanAmount", "CreditScore"]].corr())

## 4.3 Multivariate Analysis

**Three or more variables together.**

Common tools:
- Correlation heatmap
- Pair plot
- Grouped analysis
- Multivariate scatter plots

Multivariate analysis helps identify interactions and hidden patterns that may not be visible in two-variable analysis.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

sns.pairplot(df, hue="Default")
plt.show()

## 4.4 Target Variable Analysis

The target is the variable the model is expected to predict.

### Classification
Check:
- Class counts
- Class percentages
- Class imbalance

### Regression
Check:
- Mean/median
- Distribution
- Skewness
- Outliers

A highly imbalanced classification target may make accuracy misleading.

Example:

`Default = 0` → 90%  
`Default = 1` → 10%

A model predicting 0 for everyone gets 90% accuracy but is practically useless for detecting defaults.

In [ ]:
print(df["Default"].value_counts())
print("\nPercentages:")
print(df["Default"].value_counts(normalize=True) * 100)

sns.countplot(data=df, x="Default")
plt.title("Target Distribution")
plt.show()

# 5. Data Cleaning

Data cleaning improves data quality before modeling.

Main tasks:
1. Missing values
2. Duplicate data
3. Incorrect data
4. Correct data types
5. Inconsistent formats/categories

## 5.1 Missing Values

Missing values may appear as:
- `NaN`
- `None`
- Empty strings
- Special values such as `?` or `-999`

### Detection
`df.isnull().sum()`

### Treatment options
- Delete rows
- Delete columns
- Mean imputation
- Median imputation
- Mode imputation
- Forward/backward fill for suitable time-series data
- Model-based imputation

**Rule of thumb:** Median is often safer than mean when numerical data contains outliers.

In [ ]:
# Missing-value check
missing = df.isnull().sum()
print(missing)

# Example:
# df["Income"] = df["Income"].fillna(df["Income"].median())
# df["City"] = df["City"].fillna(df["City"].mode()[0])

## 5.2 Duplicate Data

Duplicate rows can cause some observations to receive excessive weight.

### Detect
`df.duplicated().sum()`

### Remove
`df.drop_duplicates()`

Always investigate duplicates before deleting them. Some repeated records may be legitimate.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicates if they are confirmed to be accidental:
# df = df.drop_duplicates()

## 5.3 Incorrect Data

Examples:
- Age = -5
- Salary = -10000
- Percentage = 250%
- Invalid category such as `"Malee"`
- Date stored incorrectly

### Example validation

In [ ]:
# Find suspicious ages
print(df[(df["Age"] < 0) | (df["Age"] > 120)])

# Find suspicious loan amounts
print(df[df["LoanAmount"] < 0])

## 5.4 Data Types

Correct data types are important for analysis and modeling.

Examples:
- Age → integer
- Salary → float
- Gender → category/object
- Date → datetime
- Target → appropriate numerical/category type

Use:
- `df.dtypes`
- `df.astype()`
- `pd.to_datetime()`

In [ ]:
print(df.dtypes)

# Examples:
# df["Age"] = df["Age"].astype("int64")
# df["Date"] = pd.to_datetime(df["Date"])

# 6. Outlier Detection & Treatment

An **outlier** is an observation that is unusually far from the rest of the data.

### Common detection methods

#### 1. Box Plot
Visual method.

#### 2. IQR Method
- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

#### 3. Z-score
A common rule is to investigate observations with absolute Z-score above about 3.

### Treatment
- Verify the value
- Keep it if it is valid
- Cap/winsorize
- Transform
- Remove only when justified

**Important:** Do not automatically delete every outlier.

In [ ]:
Q1 = df["LoanAmount"].quantile(0.25)
Q3 = df["LoanAmount"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["LoanAmount"] < lower) | (df["LoanAmount"] > upper)]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Outliers:")
display(outliers)

# 7. Feature Engineering

Feature engineering means creating better input variables from existing data.

### Examples

#### From date
`2026-08-13` → year, month, day, weekday

#### From existing numerical features
`DebtToIncome = Debt / Income`

#### From text
Text length, word count, sentiment, TF-IDF features

#### From categorical variables
Combine meaningful categories when justified.

### Goal
Create features that help the model capture useful patterns.

In [ ]:
# Example feature engineering
df["LoanToIncome"] = df["LoanAmount"] / df["Income"]

print(df[["Income", "LoanAmount", "LoanToIncome"]].head())

# 8. Feature Selection

Feature selection means choosing the most useful features for the model.

### Why?
- Reduce dimensionality
- Reduce noise
- Improve interpretability
- Reduce training time
- Potentially improve generalization

### Common approaches

**Filter methods**
- Correlation
- Chi-square
- Mutual information
- Variance threshold

**Wrapper methods**
- RFE
- Sequential feature selection

**Embedded methods**
- Lasso
- Decision tree feature importance
- Random forest importance

### Important
Do not select features using the test set. Test data should remain unseen until final evaluation.

# 9. Feature-Target Separation

Separate:
- **X** = input features
- **y** = target

Example:

`X = df.drop("Default", axis=1)`

`y = df["Default"]`

In [ ]:
X = df.drop("Default", axis=1)
y = df["Default"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", X.columns.tolist())
print("Target:", y.name)

# 10. Data Preprocessing

Preprocessing converts raw features into a form suitable for machine learning.

Typical tasks:
- Encoding categorical data
- Scaling numerical data
- Transformation
- Imputation
- Train/test split
- Building a reproducible pipeline

## 10.1 Encoding

Machine learning algorithms generally require numerical representations.

### Label Encoding
Example:

`Male → 1`  
`Female → 0`

Usually suitable for a binary target or ordinal categories when the numeric order is meaningful.

### One-Hot Encoding
Example:

`City = Pune, Mumbai, Delhi`

becomes:

- `City_Pune`
- `City_Mumbai`
- `City_Delhi`

One-hot encoding is commonly used for nominal categories where no order exists.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

cities = pd.DataFrame({"City": ["Pune", "Mumbai", "Delhi", "Pune"]})

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(cities[["City"]])

print(encoded)
print(encoder.get_feature_names_out(["City"]))

## 10.2 Feature Scaling

Scaling puts numerical features on comparable scales.

### Standardization
Formula:

`z = (x - mean) / standard_deviation`

After standardization, the feature approximately has:
- Mean = 0
- Standard deviation = 1

Use `StandardScaler`.

### Min-Max Scaling
Transforms values approximately into `[0, 1]`.

Use `MinMaxScaler`.

### Algorithms where scaling is especially important
- KNN
- K-Means
- SVM
- Logistic Regression
- Neural Networks
- PCA

Tree-based models generally do not require feature scaling.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df[["Age", "Income", "LoanAmount", "CreditScore"]])

scaled_df = pd.DataFrame(
    scaled,
    columns=["Age", "Income", "LoanAmount", "CreditScore"]
)

display(scaled_df.head())

## 10.3 Transformation

Transformation changes the distribution or representation of a feature.

Common transformations:
- Log transformation
- Square-root transformation
- Power transformation
- Yeo-Johnson
- Box-Cox

### Why transform?
Useful when:
- Data is strongly right-skewed
- Variance changes with the level of a variable
- A model benefits from a more suitable distribution

Example:

`log(Income)`

For zero/positive data, `np.log1p(x)` is often convenient because it calculates `log(1 + x)`.

In [ ]:
# Example: log transformation
df["LogIncome"] = np.log1p(df["Income"])

display(df[["Income", "LogIncome"]].head())

## 10.4 Data Leakage

**Data leakage occurs when information that would not be available at prediction time is used during model training.**

This can produce unrealistically high validation/test performance.

### Common examples

#### 1. Scaling before train/test split
Wrong:
```text
fit scaler on entire dataset → split
```

Correct:
```text
split → fit scaler on training data → transform train and test
```

#### 2. Imputing before splitting
Wrong:
```text
calculate median using train + test → fill missing values
```

Correct:
```text
split → calculate median on train → apply to train and test
```

#### 3. Target leakage
Using a feature that is created after the target event or directly contains information about the target.

### Golden rule
**Never let test-set information influence training or preprocessing decisions.**

# Final Recommended ML Workflow

A practical workflow is:

1. Define the problem and target.
2. Collect the data.
3. Understand the dataset.
4. Perform initial EDA.
5. Identify data-quality issues.
6. Split data into train/test sets at an appropriate stage.
7. Fit cleaning/preprocessing steps using training data only.
8. Perform feature engineering using information available at prediction time.
9. Select features without using the test set.
10. Encode categorical variables.
11. Scale/transform numerical variables when required.
12. Train the model.
13. Evaluate once on the untouched test set.
14. Monitor the model after deployment.

### Interview one-liner

> **EDA tells us what is in the data; data cleaning fixes quality problems; feature engineering creates useful information; feature selection keeps useful variables; preprocessing converts features into model-ready form; and data-leakage prevention ensures that evaluation is honest.**

# Quick Revision Cheat Sheet

| Topic | Key idea |
|---|---|
| Problem Definition | Define business objective, target, ML task and metric |
| Data Collection | Acquire reliable and relevant data |
| Dataset Understanding | Shape, columns, types, statistics |
| Univariate | One variable |
| Bivariate | Two variables |
| Multivariate | Multiple variables |
| Target Analysis | Understand target distribution/balance |
| Missing Values | Detect and appropriately impute/remove |
| Duplicates | Detect repeated observations |
| Incorrect Data | Fix invalid values/categories |
| Data Types | Ensure correct representation |
| Outliers | Detect and treat carefully |
| Feature Engineering | Create informative features |
| Feature Selection | Keep useful features |
| X / y Separation | Inputs vs target |
| Encoding | Convert categories to numerical representation |
| Scaling | Put numerical variables on comparable scales |
| Transformation | Change distribution/representation |
| Data Leakage | Prevent future/test information from entering training |

## 30-Minute Practice Task

Using your own CSV dataset, try this sequence:

```python
df = pd.read_csv("your_file.csv")

df.head()
df.shape
df.info()
df.describe()

df.isnull().sum()
df.duplicated().sum()

# Identify numerical and categorical columns
df.select_dtypes(include=np.number).columns
df.select_dtypes(exclude=np.number).columns

# Target analysis
df["TARGET_COLUMN"].value_counts()
```

Then perform:
**EDA → Cleaning → Outliers → Feature Engineering → Feature Selection → X/y split → Encoding → Scaling/Transformation → Leakage-safe pipeline.**